In [95]:
import pandas as pd
import numpy as np
from datetime import time

In [96]:
data = pd.read_csv("data/train.csv")

In [97]:
# stripping data

data = data.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

In [98]:
# code

invalids = [
    "nan", "NaN", "NAN", "None", "none", "NONE",
    "null", "Null", "NULL", "NA", "N/A", "n/a",
    "false", "False", "FALSE","unknown", "Unknown"
]

cols = data.columns


#
nd = data.apply(lambda col: col.isin(invalids)).any(axis=1)
data=data[~nd]
print(len(data))
data["Time_taken(min)"]=data["Time_taken(min)"].apply(lambda x:int(x.replace("(min) ","")))
data=data.sort_values("Time_taken(min)")

41368


In [99]:
# calculate IQR
def IQR(data, column):
    first = int(len(data) * 1/4 + 1)
    firstval = data[column].iloc[first]

    third = int(len(data) * 3/4 + 1)
    thirdval = data[column].iloc[third]

    iqr = thirdval - firstval

    upperfence = thirdval + (3/2) * iqr
    lowerfence = firstval - (3/2) * iqr

    return upperfence, lowerfence,thirdval
# first=int((len(data)*1/4) + 1)
# firstval=data.iloc[first,-1]

# meain=int((len(data)*1/2) + 1)
# meainval=data.iloc[meain,-1]

# third=int((len(data)*3/4) + 1)
# thirdval=data.iloc[third,-1]

# iqr=thirdval-firstval
# upperfence=thirdval+((3/2)*iqr)
# lowerfence=firstval-((3/2)*iqr)
upperfence, lowerfence,third = IQR(data, "Time_taken(min)")
data = data[(data["Time_taken(min)"] < upperfence) & (data["Time_taken(min)"] > lowerfence)]


In [100]:
time_or=data["Time_Orderd"]
time_orderd=time_or.apply(lambda time:(int(time.split(":")[0])*60 )+ (int(time.split(":")[1])))

time_pi=data["Time_Order_picked"]
time_orderd_picked=time_pi.apply(lambda time:(int(time.split(":")[0])*60 )+ (int(time.split(":")[1])))

data["Time_prep(min)"] = time_orderd_picked - time_orderd

data=data[data["Time_prep(min)"]>0]

In [101]:
time_taken=data["Time_taken(min)"]
prep_time=data["Time_prep(min)"]
data["Transit_time(min)"] = time_taken - prep_time
data=data[data["Transit_time(min)"]>0]


In [102]:
data["Total_delivery_time(min)"] = data["Time_prep(min)"]+data["Transit_time(min)"]
data = data.sort_index()

In [103]:
def time_category(time):
    hour = int(time.split(":")[0])

    if 6 <= hour < 11:
        return "Breakfast"
    elif 11 <= hour < 15:
        return "Lunch Peak"
    elif 15 <= hour < 18:
        return "Afternoon"
    elif 18 <= hour < 22:
        return "Dinner Rush"
    else:
        return "Late Night"
data["Time_Category"] = data["Time_Orderd"].apply(time_category)
data

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min),Time_prep(min),Transit_time(min),Total_delivery_time(min),Time_Category
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,...,Snack,motorcycle,0,No,Urban,24,15,9,24,Lunch Peak
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,...,Snack,scooter,1,No,Metropolitian,33,5,28,33,Dinner Rush
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,...,Drinks,motorcycle,1,No,Urban,26,15,11,26,Breakfast
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,...,Buffet,motorcycle,1,No,Metropolitian,21,10,11,21,Dinner Rush
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,...,Snack,scooter,1,No,Metropolitian,30,15,15,30,Lunch Peak
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45587,0x1178,RANCHIRES16DEL01,35,4.2,23.371292,85.327872,23.481292,85.437872,08-03-2022,21:45:00,...,Drinks,motorcycle,1,No,Metropolitian,33,10,23,33,Dinner Rush
45588,0x7c09,JAPRES04DEL01,30,4.8,26.902328,75.794257,26.912328,75.804257,24-03-2022,11:35:00,...,Meal,motorcycle,0,No,Metropolitian,32,10,22,32,Lunch Peak
45589,0xd641,AGRRES16DEL01,21,4.6,0.000000,0.000000,0.070000,0.070000,16-02-2022,19:55:00,...,Buffet,motorcycle,1,No,Metropolitian,36,15,21,36,Dinner Rush
45591,0x5eee,COIMBRES11DEL01,20,4.7,11.001753,76.986241,11.041753,77.026241,07-03-2022,13:35:00,...,Snack,motorcycle,1,No,Metropolitian,26,5,21,26,Lunch Peak


In [106]:
data=data.sort_values("Total_delivery_time(min)")
upperfence, lowerfence,third = IQR(data, "Total_delivery_time(min)")
data["Flag"] = data["Total_delivery_time(min)"] <= thirdval
data

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,...,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min),Time_prep(min),Transit_time(min),Total_delivery_time(min),Time_Category,Flag
18312,0x37dc,HYDRES20DEL02,28,4.8,17.483216,78.552111,17.613216,78.682111,04-04-2022,23:30:00,...,motorcycle,1,No,Urban,10,5,5,10,Late Night,True
39712,0x3a07,PUNERES14DEL01,31,4.9,18.516216,73.842527,18.526216,73.852527,13-03-2022,10:15:00,...,scooter,1,No,Urban,10,5,5,10,Breakfast,True
8013,0x3a71,RANCHIRES03DEL03,25,4.9,0.000000,0.000000,0.060000,0.060000,03-04-2022,22:50:00,...,scooter,0,No,Metropolitian,10,5,5,10,Late Night,True
17615,0x28ca,HYDRES18DEL02,27,4.7,17.450851,78.379347,17.580851,78.509347,16-03-2022,22:20:00,...,electric_scooter,0,No,Urban,10,5,5,10,Late Night,True
20008,0x465d,MUMRES16DEL03,23,4.9,19.120083,72.907385,19.150083,72.937385,28-03-2022,20:15:00,...,scooter,0,No,Metropolitian,10,5,5,10,Dinner Rush,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7085,0x581d,JAPRES08DEL03,28,3.6,26.910262,75.783013,26.940262,75.813013,05-04-2022,21:40:00,...,motorcycle,1,No,Metropolitian,53,10,43,53,Dinner Rush,False
3764,0x2f9f,MYSRES010DEL01,21,4.1,12.321214,76.621094,12.361214,76.661094,01-03-2022,13:45:00,...,motorcycle,3,No,Metropolitian,53,10,43,53,Lunch Peak,False
44789,0xd0f,SURRES02DEL02,31,4,21.186608,72.794136,21.316608,72.924136,18-03-2022,20:35:00,...,motorcycle,0,Yes,Urban,53,5,48,53,Dinner Rush,False
34677,0xdf38,AGRRES19DEL02,34,5,0.000000,0.000000,0.080000,0.080000,12-02-2022,21:35:00,...,motorcycle,3,Yes,Metropolitian,53,15,38,53,Dinner Rush,False
